# DocQWise v0.2.0 — Getting Started

## Read. Extract. Retrieve.

Document intelligence that adapts, accelerates, and scales.

### Install

```bash
pip install docqwise
```

### Optional (for AI-powered extraction)

```bash
# Ollama (local LLM — recommended)
# Download from https://ollama.com
ollama pull nemotron-mini

# OR HuggingFace (local GPU)
pip install transformers torch bitsandbytes accelerate

# For RAG extraction
pip install sentence-transformers
```

---

## 1. Setup and verify

In [ ]:
from docqwise import Docqwise, __version__
import shutil, os, tempfile

print(f"DocQWise v{__version__}")

# Create engine with temp storage
store = os.path.join(tempfile.gettempdir(), "nb_docqwise")
if os.path.exists(store):
    shutil.rmtree(store, ignore_errors=True)

dq = Docqwise(store_path=store)
print(dq)
print(dq.metrics())

## 2. Ingest documents

Read any format — PDF, CSV, JSON, DOCX, Excel, images, and more.
Incremental by default: re-ingest skips unchanged files.

In [ ]:
# Ingest multiple formats
for source in ["demo/sample_invoice.pdf", "demo/sample_sales.csv", "demo/sample_data.json"]:
    r = dq.ingest(source, embed=False)
    print(f"{os.path.basename(source):25s} processed={r['processed']} time={r['elapsed_s']}s")

# Re-ingest skips unchanged files
r = dq.ingest("demo/sample_invoice.pdf", embed=False)
print(f"
Re-ingest: processed={r['processed']} (0 = skipped, file unchanged)")

## 3. Extract fields

Use templates (invoice, contract, resume, receipt) or define custom schema.

In [ ]:
# Invoice template extraction
result = dq.extract_fields("demo/sample_invoice.pdf", template="invoice", method="regex")

print(f"Strategy: {result.strategy_used}")
print(f"Confidence: {result.confidence:.2f}
")

for name, field in result.fields.items():
    print(f"{name:20s} = {str(field.value):25s} [{field.confidence:.2f}]")

print(f"
JSON:
{result.to_json()}")

## 4. Custom schema

Define exactly what fields you want to extract.

## 5. Custom prompts

You design the prompts. We run the pipeline.
Use `{context}` for document text and `{schema}` for schema placeholder.

In [ ]:
# Your own prompt — full control over what LLM does
result = dq.extract_fields("demo/sample_invoice.pdf",
    prompt="""You are a financial auditor reviewing this invoice.
Extract the vendor name, total amount, and check if tax is calculated correctly.

Document:
{context}

Return JSON: {{"vendor": "...", "total": ..., "tax": ..., "tax_correct": true/false}}
JSON:""",
)
print(f"Strategy: {result.strategy_used}")
for name, field in result.fields.items():
    print(f"{name:20s} = {field.value}")

In [ ]:
# Your prompt template with schema placeholder
result = dq.extract_fields("demo/sample_contract.pdf",
    schema={
        "parties": {"type": "array", "description": "All parties involved"},
        "risk_level": {"type": "string", "description": "Low, Medium, or High"},
    },
    prompt_template="""You are a legal analyst.
Extract these fields from the contract:
{schema}

Contract:
{context}

Return JSON with the field names as keys.
JSON:""",
)
print(f"Strategy: {result.strategy_used}")
for name, field in result.fields.items():
    print(f"{name:20s} = {str(field.value)[:40]}")

In [ ]:
schema = {
    "sender": {"type": "string", "description": "Who sent this document"},
    "total_value": {"type": "number", "description": "Total monetary value"},
    "payment_due": {"type": "date", "description": "When payment is due"},
}

result = dq.extract_fields("demo/sample_invoice.pdf", schema=schema, method="regex")
for name, field in result.fields.items():
    print(f"{name:20s} = {field.value}")

## 6. Self-improving corrections

Fix a wrong field once. Stored as exact override. Applied automatically on future similar documents.
No ML retraining. Deterministic.

In [ ]:
result = dq.extract_fields("demo/sample_invoice.pdf", template="invoice", method="regex")

# Correct wrong or missing fields
result.correct({"tax": 33300.00, "gst_number": "29AABCU9603R1ZM"})

print(f"tax = {result.fields['tax'].value} (corrected: {result.fields['tax'].is_corrected})")
print(f"gst = {result.fields['gst_number'].value} (new field added)")
print(f"
Corrections stored: {dq.learning_report()['total_corrections']}")
print("Next similar document will auto-apply these corrections.")

## 7. Entity extraction

In [ ]:
entities = dq.extract_entities("demo/sample_invoice.pdf", method="regex")
for e in entities:
    print(f"[{e.entity_type:10s}] {e.text}")

## 8. Document classification

In [ ]:
for doc_path in ["demo/sample_invoice.pdf", "demo/sample_contract.pdf"]:
    labels = dq.classify(doc_path)
    top = labels[0]
    print(f"{os.path.basename(doc_path):25s} → {top['label']} ({top['confidence']:.0%})")

## 9. Document comparison

In [ ]:
diff = dq.compare("demo/sample_invoice.pdf", "demo/sample_contract.pdf")
print(f"Similarity: {diff['similarity']:.1%}")
print(f"Additions: {diff['additions_count']}, Deletions: {diff['deletions_count']}")

## 10. Structured data Q&A

Ask questions about CSV, Excel, or database data.
Uses exact computation (SUM, AVG, COUNT) — not LLM guessing.

In [ ]:
for q in ["What is the total amount?", "What is the average amount?",
          "How many items are there?", "What is the maximum amount?"]:
    answer = dq.ask(q, source="demo/sample_sales.csv")
    print(f"Q: {q}")
    print(f"A: {answer}
")

## 11. Schema detection

In [ ]:
schema = dq.detect_schema("demo/sample_sales.csv")
for f in schema["fields"]:
    print(f"{f['name']:15s} type={f['type']:10s} nullable={f['nullable']}")

## 12. PII detection

In [ ]:
pii = dq.detect_pii("demo/sample_invoice.pdf")
for p in pii:
    print(f"[{p.pii_type:12s}] {p.value}")

## 13. Pipeline DAG

In [ ]:
from docqwise.pipeline.pipeline import Pipeline

pipe = Pipeline("extraction_pipeline")
pipe.add_node("read", node_type="reader")
pipe.add_node("extract", node_type="extractor")
pipe.add_node("store", node_type="store")
pipe.connect("read", "extract")
pipe.connect("extract", "store")
print(f"Valid: {pipe.validate()}")
print(pipe.visualize())

## 14. Document graph

In [ ]:
from docqwise.graph.document_graph import DocumentGraph

graph = DocumentGraph()
graph.add_document("invoice", {"type": "invoice"})
graph.add_document("contract", {"type": "contract"})
graph.add_entity("acme", "ORG", "Acme Corporation")
graph.add_edge("invoice", "acme", "from_vendor")
graph.add_edge("contract", "acme", "with_party")
print(f"Nodes: {graph.node_count}, Edges: {graph.edge_count}")
print(f"Communities: {graph.communities()}")
print(f"Query: {graph.query('Acme')}")

## 15. Engine configurations

Same API. Different backends. User picks their stack.

In [ ]:
print("Default:    ", Docqwise())
print("Production: ", Docqwise(strategy="hybrid", workers=8, gpu_workers=2))
print("Local-only: ", Docqwise(strategy="local_only", workers=4))
print("Custom DB:  ", Docqwise(vector_store="qdrant", search_mode="hybrid"))

## 16. Available methods and templates

In [ ]:
from docqwise.factory import ExtractorFactory, TemplateFactory, LLMFactory

print(f"Extraction methods: {ExtractorFactory.available_methods()}")
print(f"Templates:          {TemplateFactory.available()}")
print(f"LLM backends:       {LLMFactory.available()}")

## Cleanup

In [ ]:
shutil.rmtree(os.path.join(tempfile.gettempdir(), "nb_docqwise"), ignore_errors=True)
print("Done!")

---

## Next steps

| What | How |
|---|---|
| AI extraction with Ollama | `python demo/02_ollama.py` |
| AI extraction with HuggingFace | `python demo/03_huggingface.py` |
| RAG-based extraction | `python demo/04_rag.py` |
| REST API server | `docqwise serve --port 8000` |
| Docker | `docker compose up -d` |

## Ecosystem

| Library | Tagline | Domain |
|---|---|---|
| SightRAG | See. Search. Retrieve. | Visual intelligence |
| sonarwise | Hear. Search. Retrieve. | Audio intelligence |
| **docqwise** | **Read. Extract. Retrieve.** | **Document intelligence** |
| adaptive-intelligence | Learn. Remember. Adapt. | Orchestration |
| llmevalkit | Evaluate. Score. Improve. | Evaluation |

---

**DocQWise v0.2.0** — Read. Extract. Retrieve.

*Sometimes the ant carries the elephant.* 🐜